In [1]:
# colab_01_veri_hazirlik.py  (v3 - yillik lag eklendi)
# Rossmann — 1115 magaza, 28 gun ileri tahmin icin veri hazirlik
# Cikti: NN tensorleri (.npz) + XGBoost duz tablo (.parquet) + meta (.json)

import os, gc, json, collections, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [2]:
# ------------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------------
VERI_DIZIN = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman"
CIKTI_DIZIN = f"{VERI_DIZIN}/hazirlik"

GECMIS = 56           # girdi penceresi (gun) - 8 hafta
UFUK = 28             # tahmin ufku (gun) - 4 hafta
YIL_LAG = 364         # 52*7 -> haftanin gunu korunur (365 DEGIL)
ORIGIN_ADIM = 3       # 7 ile aralarinda asal -> origin'ler tum haftaya yayilir
XGB_ORIGIN_ATLA = 3   # XGBoost icin her 3. origin (3x3=9 gun)

TEST_ORIGIN = pd.Timestamp("2015-07-03")   # hedef: 07-04 .. 07-31
VAL_ORIGIN = pd.Timestamp("2015-06-05")    # hedef: 06-06 .. 07-03
VAL_GERI = (0, 3, 6, 9, 12)

os.makedirs(CIKTI_DIZIN, exist_ok=True)

In [3]:
# ------------------------------------------------------------------
# 2. DRIVE + OKUMA
# ------------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Colab disi ortam, drive mount atlandi.")

train = pd.read_csv(f"{VERI_DIZIN}/train.csv", parse_dates=["Date"], low_memory=False)
store = pd.read_csv(f"{VERI_DIZIN}/store.csv")

print("train:", train.shape, "| store:", store.shape)
print("tarih araligi:", train.Date.min().date(), "->", train.Date.max().date())

# test.csv / sample_submission.csv KULLANILMIYOR:
# Kaggle yarisma donemi (2015-08 sonrasi), Sales kolonu bos -> metrik olculemez.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: (1017209, 9) | store: (1115, 10)
tarih araligi: 2013-01-01 -> 2015-07-31


In [4]:
# ------------------------------------------------------------------
# 3. TEMIZLIK
# ------------------------------------------------------------------
train["StateHolidayFlag"] = (train["StateHoliday"].astype(str) != "0").astype(np.int8)
train["Open"] = train["Open"].fillna(0).astype(np.int8)
train["Promo"] = train["Promo"].astype(np.int8)
train["SchoolHoliday"] = train["SchoolHoliday"].astype(np.int8)
train["Sales"] = train["Sales"].astype(np.float32)

sifir_acik = int(((train["Open"] == 1) & (train["Sales"] <= 0)).sum())
train.loc[(train["Open"] == 1) & (train["Sales"] <= 0), "Open"] = 0
print(f"Acik ama sifir ciro (kapali sayildi): {sifir_acik}")

# Customers KULLANILMIYOR - sizinti: tahmin aninda bilinmez
train = train.drop(columns=[c for c in ["Customers"] if c in train.columns])

magazalar = np.sort(train["Store"].unique())
tarihler = pd.date_range(train["Date"].min(), train["Date"].max(), freq="D")
n_magaza, n_gun = len(magazalar), len(tarihler)
print(f"magaza: {n_magaza} | gun: {n_gun}")

gun_idx = {d: i for i, d in enumerate(tarihler)}

Acik ama sifir ciro (kapali sayildi): 54
magaza: 1115 | gun: 942


In [5]:
# ------------------------------------------------------------------
# 4. GUN x MAGAZA MATRISLERI
# ------------------------------------------------------------------
def matris_yap(kolon):
    m = train.pivot_table(index="Store", columns="Date", values=kolon, aggfunc="first")
    return m.reindex(index=magazalar, columns=tarihler).values.astype(np.float32)

SATIS = matris_yap("Sales")
ACIK = matris_yap("Open")
PROMO = matris_yap("Promo")
OKUL = matris_yap("SchoolHoliday")
TATIL = matris_yap("StateHolidayFlag")

eksik = np.isnan(SATIS)
print(f"eksik magaza-gun: {eksik.sum():,} ({100*eksik.mean():.2f}%)")
SATIS[eksik] = 0.0
ACIK[np.isnan(ACIK)] = 0.0

for M in (PROMO, OKUL, TATIL):
    gun_ort = np.nan_to_num(np.round(np.nanmean(M, axis=0)))
    nan_yer = np.isnan(M)
    M[nan_yer] = np.broadcast_to(gun_ort, M.shape)[nan_yer]

ACIK = ACIK.astype(np.int8)
PROMO = PROMO.astype(np.int8)
OKUL = OKUL.astype(np.int8)
TATIL = TATIL.astype(np.int8)

bosluk_magaza = magazalar[eksik.sum(axis=1) > 30]
print(f"30 gunden fazla veri boslugu olan magaza: {len(bosluk_magaza)}")

del train; gc.collect()

eksik magaza-gun: 33,121 (3.15%)
30 gunden fazla veri boslugu olan magaza: 180


0

In [6]:
# ------------------------------------------------------------------
# 5. MAGAZA STATIK OZELLIKLERI
# ------------------------------------------------------------------
st = store.set_index("Store").reindex(magazalar).copy()
st["StoreTypeKod"] = st["StoreType"].astype("category").cat.codes.astype(np.int8)
st["AssortmentKod"] = st["Assortment"].astype("category").cat.codes.astype(np.int8)
st["CompetitionDistance"] = st["CompetitionDistance"].fillna(st["CompetitionDistance"].median())
st["LogRakipMesafe"] = np.log1p(st["CompetitionDistance"]).astype(np.float32)
st["Promo2"] = st["Promo2"].fillna(0).astype(np.int8)

rakip_yil = st["CompetitionOpenSinceYear"].fillna(0).astype(int).values
rakip_ay_no = st["CompetitionOpenSinceMonth"].fillna(1).astype(int).values
RAKIP_AY_MUTLAK = np.where(rakip_yil > 1900, rakip_yil*12 + rakip_ay_no, -99999).astype(np.int32)

def promo2_baslangic(r):
    if r["Promo2"] == 0 or pd.isna(r["Promo2SinceYear"]):
        return pd.NaT
    return pd.Timestamp(f"{int(r['Promo2SinceYear'])}-01-01") + \
           pd.Timedelta(weeks=int(r["Promo2SinceWeek"]) - 1)
st["Promo2Baslangic"] = st.apply(promo2_baslangic, axis=1)
PROMO2_BAS_GUN = np.array([
    (b - pd.Timestamp("1970-01-01")).days if pd.notna(b) else 999999
    for b in st["Promo2Baslangic"]], dtype=np.int64)

AY_KISA = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
           7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
aylar_kisa = np.array([AY_KISA[d.month] for d in tarihler])

PROMO2 = np.zeros((n_magaza, n_gun), dtype=np.int8)
for i, s in enumerate(magazalar):
    if st.loc[s, "Promo2"] == 0 or pd.isna(st.loc[s, "Promo2Baslangic"]):
        continue
    aralik = str(st.loc[s, "PromoInterval"])
    if aralik in ("nan", "None"):
        continue
    basladi = tarihler >= st.loc[s, "Promo2Baslangic"]
    ay_uygun = np.isin(aylar_kisa, list(set(aralik.split(","))))
    PROMO2[i] = (basladi & ay_uygun).astype(np.int8)

In [7]:
# ------------------------------------------------------------------
# 6. ORIGIN LISTELERI  (val ONCE, egitim ondan tureyecek)
# ------------------------------------------------------------------
val_originler = sorted(VAL_ORIGIN - pd.Timedelta(days=k) for k in VAL_GERI)
test_originler = [TEST_ORIGIN]

# KRITIK 1: egitim hedefleri en erken val hedefine DEGMEMELI
son_egitim_origin = min(val_originler) - pd.Timedelta(days=UFUK)

# KRITIK 2: YIL_LAG icin hedef gunun 364 gun oncesi veride OLMALI
#   ilk hedef gun = origin+1 -> (origin+1) - 364 >= 0
ilk_origin_idx = GECMIS - 1
print(f"ilk gecerli origin: {tarihler[ilk_origin_idx].date()} "
      f"(yillik lag {YIL_LAG} gun oncesi yoksa kanal 0 + bayrak 0)")

egitim_originler = [d for d in tarihler[ilk_origin_idx::ORIGIN_ADIM] if d <= son_egitim_origin]

dagilim = collections.Counter(d.dayofweek for d in egitim_originler)
print(f"\negitim origin: {len(egitim_originler)} "
      f"({egitim_originler[0].date()} -> {egitim_originler[-1].date()})")
print("origin gun dagilimi (0=Pzt):", dict(sorted(dagilim.items())))
assert len(dagilim) == 7, "Origin'ler haftanin tum gunlerine yayilmadi!"

print(f"egitim son hedef: {(son_egitim_origin + pd.Timedelta(days=UFUK)).date()}")
print(f"val  hedef: {(min(val_originler)+pd.Timedelta(days=1)).date()} -> "
      f"{(max(val_originler)+pd.Timedelta(days=UFUK)).date()}  ({len(val_originler)} origin)")
print(f"test hedef: {(TEST_ORIGIN+pd.Timedelta(days=1)).date()} -> "
      f"{(TEST_ORIGIN+pd.Timedelta(days=UFUK)).date()}")

ilk gecerli origin: 2013-02-25 (yillik lag 364 gun oncesi yoksa kanal 0 + bayrak 0)

egitim origin: 264 (2013-02-25 -> 2015-04-25)
origin gun dagilimi (0=Pzt): {0: 38, 1: 37, 2: 38, 3: 38, 4: 37, 5: 38, 6: 38}
egitim son hedef: 2015-05-24
val  hedef: 2015-05-25 -> 2015-07-03  (5 origin)
test hedef: 2015-07-04 -> 2015-07-31


In [8]:
# ------------------------------------------------------------------
# 7. OLCEKLEME (sadece egitim doneminden)
# ------------------------------------------------------------------
olcek_son_idx = gun_idx[min(val_originler)] + 1
LOG_SATIS = np.log1p(SATIS)
maske_e = (ACIK[:, :olcek_son_idx] == 1)

magaza_ort = np.zeros(n_magaza, dtype=np.float32)
magaza_std = np.ones(n_magaza, dtype=np.float32)
for i in range(n_magaza):
    v = LOG_SATIS[i, :olcek_son_idx][maske_e[i]]
    if v.size > 1:
        magaza_ort[i], magaza_std[i] = v.mean(), v.std()
    elif v.size == 1:
        magaza_ort[i] = v.mean()
magaza_std[magaza_std < 1e-3] = 1.0
print(f"olcekleme kesim tarihi: {tarihler[olcek_son_idx-1].date()}")

LOG_OLCEK = ((LOG_SATIS - magaza_ort[:, None]) / magaza_std[:, None]).astype(np.float32)
LOG_OLCEK[ACIK == 0] = 0.0

olcekleme kesim tarihi: 2015-05-24


In [9]:
# ------------------------------------------------------------------
# 8. NN TENSORLERI
# ------------------------------------------------------------------
GECMIS_KANAL = ["satis_olcek", "acik", "promo", "okul", "tatil"]
GELECEK_KANAL = ["acik", "promo", "okul", "tatil", "promo2",
                 "dow_sin", "dow_cos", "ay_sin", "ay_cos",
                 "dom_sin", "dom_cos", "h_norm",
                 "gecen_yil_olcek", "gecen_yil_acik"]   # YENI: yillik lag
IDX_GY = GELECEK_KANAL.index("gecen_yil_olcek")
IDX_GY_ACIK = GELECEK_KANAL.index("gecen_yil_acik")

h_norm = (np.arange(1, UFUK + 1) / UFUK).astype(np.float32)

def gelecek_takvim(o_idx):
    g = tarihler[o_idx + 1: o_idx + 1 + UFUK]
    dow, ay, dom = g.dayofweek.values, g.month.values, g.day.values
    return np.stack([
        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7),
        np.sin(2*np.pi*ay/12), np.cos(2*np.pi*ay/12),
        np.sin(2*np.pi*dom/31), np.cos(2*np.pi*dom/31)], axis=1).astype(np.float32)

def tensor_uret(originler, filtrele):
    N = len(originler) * n_magaza
    X_g = np.zeros((N, GECMIS, len(GECMIS_KANAL)), dtype=np.float32)
    X_f = np.zeros((N, UFUK, len(GELECEK_KANAL)), dtype=np.float32)
    X_s = np.zeros(N, dtype=np.int32)
    y = np.zeros((N, UFUK), dtype=np.float32)
    y_ham = np.zeros((N, UFUK), dtype=np.float32)
    maske = np.zeros((N, UFUK), dtype=np.float32)
    org_idx = np.zeros(N, dtype=np.int32)

    p = 0
    for o in originler:
        oi = gun_idx[o]
        g = slice(oi - GECMIS + 1, oi + 1)
        f_idx = np.arange(oi + 1, oi + 1 + UFUK)
        tak = gelecek_takvim(oi)
        b = slice(p, p + n_magaza)

        X_g[b, :, 0] = LOG_OLCEK[:, g]
        X_g[b, :, 1] = ACIK[:, g]
        X_g[b, :, 2] = PROMO[:, g]
        X_g[b, :, 3] = OKUL[:, g]
        X_g[b, :, 4] = TATIL[:, g]

        X_f[b, :, 0] = ACIK[:, f_idx]
        X_f[b, :, 1] = PROMO[:, f_idx]
        X_f[b, :, 2] = OKUL[:, f_idx]
        X_f[b, :, 3] = TATIL[:, f_idx]
        X_f[b, :, 4] = PROMO2[:, f_idx]
        X_f[b, :, 5:11] = tak[None, :, :]
        X_f[b, :, 11] = h_norm

        # --- YILLIK LAG: hedef gunun 364 gun oncesi ---
        gy = f_idx - YIL_LAG
        ok = gy >= 0
        if ok.any():
            X_f[b][:, ok, IDX_GY] = LOG_OLCEK[:, gy[ok]]
            X_f[b][:, ok, IDX_GY_ACIK] = ACIK[:, gy[ok]]

        X_s[b] = np.arange(n_magaza)
        y[b] = LOG_OLCEK[:, f_idx]
        y_ham[b] = SATIS[:, f_idx]
        maske[b] = (ACIK[:, f_idx] == 1).astype(np.float32)
        org_idx[b] = oi
        p += n_magaza

    d = dict(X_gecmis=X_g, X_gelecek=X_f, X_magaza=X_s, y=y, y_ham=y_ham,
             maske=maske, origin_idx=org_idx, magaza_no=magazalar[X_s])
    if filtrele:
        ok2 = (X_g[:, :, 1].sum(axis=1) > 7) & (maske.sum(axis=1) > 0)
        d = {k: v[ok2] for k, v in d.items()}
        print(f"  filtre: {N:,} -> {int(ok2.sum()):,}")
    return d

print("\n--- NN tensorleri ---")
for ad, orgs, filt in [("egitim", egitim_originler, True),
                       ("val", val_originler, False),
                       ("test", test_originler, False)]:
    d = tensor_uret(orgs, filt)
    gy_dolu = float((d["X_gelecek"][:, :, IDX_GY] != 0).mean())
    np.savez_compressed(f"{CIKTI_DIZIN}/nn_{ad}.npz", **d)
    print(f"nn_{ad}.npz -> X_gecmis {d['X_gecmis'].shape} | "
          f"X_gelecek {d['X_gelecek'].shape} | "
          f"gecerli hedef {int(d['maske'].sum()):,} | yil_lag dolu %{100*gy_dolu:.1f}")
    del d; gc.collect()


--- NN tensorleri ---
  filtre: 294,360 -> 282,472
nn_egitim.npz -> X_gecmis (282472, 56, 5) | X_gelecek (282472, 28, 14) | gecerli hedef 6,536,370 | yil_lag dolu %50.8
nn_val.npz -> X_gecmis (5575, 56, 5) | X_gelecek (5575, 28, 14) | gecerli hedef 130,570 | yil_lag dolu %78.3
nn_test.npz -> X_gecmis (1115, 56, 5) | X_gelecek (1115, 28, 14) | gecerli hedef 26,845 | yil_lag dolu %71.8


In [10]:
# ------------------------------------------------------------------
# 9. XGBOOST DUZ TABLO
# ------------------------------------------------------------------
def pencere_ort(oi, uzunluk):
    g = slice(oi - uzunluk + 1, oi + 1)
    s, a = SATIS[:, g], ACIK[:, g]
    n = a.sum(axis=1)
    return np.where(n > 0, (s*a).sum(axis=1)/np.maximum(n, 1), 0.0).astype(np.float32)

def dow_ort_tablo(oi, hafta=4):
    out = np.zeros((n_magaza, 7), dtype=np.float32)
    for dw in range(7):
        idxler = [oi - k for k in range(1, hafta*7 + 1)
                  if oi - k >= 0 and tarihler[oi - k].dayofweek == dw]
        if not idxler:
            continue
        s, a = SATIS[:, idxler], ACIK[:, idxler]
        n = a.sum(axis=1)
        out[:, dw] = np.where(n > 0, (s*a).sum(axis=1)/np.maximum(n, 1), 0.0)
    return out

def duz_tablo(originler):
    parcalar = []
    for o in originler:
        oi = gun_idx[o]
        t_idx = np.arange(oi + 1, oi + 1 + UFUK)
        t_gun = tarihler[t_idx]

        ort7, ort28, ort56 = pencere_ort(oi, 7), pencere_ort(oi, 28), pencere_ort(oi, 56)
        g28 = slice(oi - 27, oi + 1)
        std28 = np.where(ACIK[:, g28].sum(axis=1) > 1,
                         SATIS[:, g28].std(axis=1), 0.0).astype(np.float32)
        acik_oran28 = ACIK[:, g28].mean(axis=1).astype(np.float32)
        dow_tab = dow_ort_tablo(oi)

        dow = t_gun.dayofweek.values
        ay = t_gun.month.values
        gun = t_gun.day.values
        hafta_no = t_gun.isocalendar().week.values.astype(np.int8)
        t_ay_mutlak = (t_gun.year.values*12 + ay).astype(np.int32)
        t_epoch = (t_gun - pd.Timestamp("1970-01-01")).days.values.astype(np.int64)

        gy_idx = t_idx - YIL_LAG
        gy_ok = gy_idx >= 0
        gecen_yil = np.zeros((n_magaza, UFUK), dtype=np.float32)
        gecen_yil_acik = np.zeros((n_magaza, UFUK), dtype=np.int8)
        gecen_yil[:, gy_ok] = SATIS[:, gy_idx[gy_ok]]
        gecen_yil_acik[:, gy_ok] = ACIK[:, gy_idx[gy_ok]]

        rakip_ay = np.where(RAKIP_AY_MUTLAK[:, None] > -99999,
                            t_ay_mutlak[None, :] - RAKIP_AY_MUTLAK[:, None],
                            -1).astype(np.float32)
        promo2_hafta = np.where(PROMO2_BAS_GUN[:, None] < 999999,
                                (t_epoch[None, :] - PROMO2_BAS_GUN[:, None])/7.0,
                                -1).astype(np.float32)
        promo2_hafta[promo2_hafta < 0] = -1

        bir = np.ones((n_magaza, UFUK), dtype=np.float32)
        parcalar.append(pd.DataFrame({
            "magaza": np.repeat(magazalar, UFUK),
            "tarih": np.tile(t_gun.values, n_magaza),
            "h": np.tile(np.arange(1, UFUK+1, dtype=np.int16), n_magaza),
            "dow": np.tile(dow.astype(np.int8), n_magaza),
            "ay": np.tile(ay.astype(np.int8), n_magaza),
            "gun": np.tile(gun.astype(np.int8), n_magaza),
            "hafta": np.tile(hafta_no, n_magaza),
            "acik": ACIK[:, t_idx].ravel(),
            "promo": PROMO[:, t_idx].ravel(),
            "okul": OKUL[:, t_idx].ravel(),
            "tatil": TATIL[:, t_idx].ravel(),
            "promo2": PROMO2[:, t_idx].ravel(),
            "magaza_tipi": np.repeat(st["StoreTypeKod"].values, UFUK),
            "urun_yelpazesi": np.repeat(st["AssortmentKod"].values, UFUK),
            "log_rakip_mesafe": np.repeat(st["LogRakipMesafe"].values, UFUK),
            "rakip_ay": rakip_ay.ravel(),
            "promo2_hafta": promo2_hafta.ravel(),
            "ort7": (ort7[:, None]*bir).ravel(),
            "ort28": (ort28[:, None]*bir).ravel(),
            "ort56": (ort56[:, None]*bir).ravel(),
            "std28": (std28[:, None]*bir).ravel(),
            "acik_oran28": (acik_oran28[:, None]*bir).ravel(),
            "dow_ort": dow_tab[:, dow].ravel(),
            "gecen_yil": gecen_yil.ravel(),
            "gecen_yil_acik": gecen_yil_acik.ravel(),
            "hedef": SATIS[:, t_idx].ravel(),
        }))
    df = pd.concat(parcalar, ignore_index=True)
    return df[df["acik"] == 1].drop(columns=["acik"]).reset_index(drop=True)

print("\n--- XGBoost tablolari ---")
xgb_egitim_originler = egitim_originler[::XGB_ORIGIN_ATLA]
xd = collections.Counter(d.dayofweek for d in xgb_egitim_originler)
print(f"xgb egitim origin: {len(xgb_egitim_originler)} | gun dagilimi: {dict(sorted(xd.items()))}")
assert len(xd) == 7, "XGB origin'leri haftaya yayilmadi!"

for ad, orgs in [("egitim", xgb_egitim_originler),
                 ("val", val_originler), ("test", test_originler)]:
    df = duz_tablo(orgs)
    df.to_parquet(f"{CIKTI_DIZIN}/xgb_{ad}.parquet", index=False)
    print(f"xgb_{ad}.parquet -> {df.shape}  ({df.tarih.min().date()} .. {df.tarih.max().date()})")
    del df; gc.collect()



--- XGBoost tablolari ---
xgb egitim origin: 88 | gun dagilimi: {0: 13, 1: 12, 2: 13, 3: 12, 4: 13, 5: 12, 6: 13}
xgb_egitim.parquet -> (2191495, 25)  (2013-02-26 .. 2015-05-17)
xgb_val.parquet -> (130570, 25)  (2015-05-25 .. 2015-07-03)
xgb_test.parquet -> (26845, 25)  (2015-07-04 .. 2015-07-31)


In [11]:
# ------------------------------------------------------------------
# 10. META
# ------------------------------------------------------------------
meta = {
    "gecmis": GECMIS, "ufuk": UFUK, "yil_lag": YIL_LAG,
    "origin_adim": ORIGIN_ADIM, "xgb_origin_atla": XGB_ORIGIN_ATLA,
    "n_magaza": int(n_magaza),
    "magazalar": magazalar.tolist(),
    "magaza_ort": magaza_ort.tolist(),
    "magaza_std": magaza_std.tolist(),
    "gecmis_kanal": GECMIS_KANAL,
    "gelecek_kanal": GELECEK_KANAL,
    "promo_kanal_idx": GELECEK_KANAL.index("promo"),
    "okul_kanal_idx": GELECEK_KANAL.index("okul"),
    "tarihler": [str(d.date()) for d in tarihler],
    "ilk_egitim_origin": str(egitim_originler[0].date()),
    "egitim_son_hedef": str((son_egitim_origin + pd.Timedelta(days=UFUK)).date()),
    "val_hedef": [str((min(val_originler)+pd.Timedelta(days=1)).date()),
                  str((max(val_originler)+pd.Timedelta(days=UFUK)).date())],
    "test_hedef": [str((TEST_ORIGIN+pd.Timedelta(days=1)).date()),
                   str((TEST_ORIGIN+pd.Timedelta(days=UFUK)).date())],
    "test_origin": str(TEST_ORIGIN.date()),
    "olcekleme_kesim": str(tarihler[olcek_son_idx-1].date()),
    "origin_gun_dagilimi": {int(k): int(v) for k, v in sorted(dagilim.items())},
    "bosluklu_magazalar": bosluk_magaza.tolist(),
    "acik_sifir_ciro_satir": sifir_acik,
    "eksik_magaza_gun": int(eksik.sum()),
}
with open(f"{CIKTI_DIZIN}/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\nBitti ->", CIKTI_DIZIN)


Bitti -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/hazirlik


In [12]:
import pandas as pd
tr = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/datasets/rossman/xgb_egitim.parquet')
te = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/datasets/rossman/xgb_test.parquet')
print("egitim son tarih:", tr.tarih.max().date())
print("test  ilk tarih :", te.tarih.min().date())
print("kesisim:", len(set(tr.tarih) & set(te.tarih)))

egitim son tarih: 2015-06-04
test  ilk tarih : 2015-07-04
kesisim: 0


In [13]:
import pandas as pd, json
D = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman/hazirlik"
m = json.load(open(f"{D}/meta.json"))
tr = pd.read_parquet(f"{D}/xgb_egitim.parquet", columns=["tarih"])
va = pd.read_parquet(f"{D}/xgb_val.parquet", columns=["tarih"])
te = pd.read_parquet(f"{D}/xgb_test.parquet", columns=["tarih"])

print("egitim-val kesisim :", len(set(tr.tarih) & set(va.tarih)))   # 0 olmali
print("egitim-test kesisim:", len(set(tr.tarih) & set(te.tarih)))   # 0 olmali
print("val-test kesisim   :", len(set(va.tarih) & set(te.tarih)))   # 0 olmali
print("olcekleme kesim    :", m["olcekleme_kesim"])
print("origin gun dagilimi:", m["origin_gun_dagilimi"])

egitim-val kesisim : 0
egitim-test kesisim: 0
val-test kesisim   : 0
olcekleme kesim    : 2015-05-24
origin gun dagilimi: {'0': 38, '1': 37, '2': 38, '3': 38, '4': 37, '5': 38, '6': 38}
